# 03. The Differentiability Problem in Keypoint Fine-Tuning
During the development of this pipeline, an attempt was made to fine-tune the LoFTR Transformer layers using a custom Identity Loss. Since Sentinel-2 images are perfectly ortho-rectified, the hypothesis was simple: coordinate $(X, Y)$ on the `Before` tile should map exactly to $(X, Y)$ on the `After` tile. Any deviation is considered a matching error.

However, during training, **the loss remained completely frozen at `0.0000` across all epochs**.

### Why did the gradients vanish?
The root cause lies in **non-differentiable operations**. To compute the physical distance between matched points, we must extract their exact coordinates from the network's output. Under the hood, extracting discrete coordinates from continuous dense correlation matrices requires operations like `argmax` (finding the index of the highest probability pixel on a heatmap).

In PyTorch (and neural networks in general), **`argmax` is a non-differentiable step**. When the backward pass calculates gradients, it hits the `argmax` operation, the chain rule breaks, and the gradients drop to zero. Consequently, the optimizer takes steps, but the Transformer weights receive no actual updates.

### The Solution for Future Development
To properly fine-tune such a model, we cannot use hard coordinate extraction (`argmax`). Instead, we must compute the loss directly on the **dense probability correlation matrices** (using Cross-Entropy or Focal Loss) or apply a **Soft-Argmax** technique, which maintains the computational graph and allows gradients to flow back into the attention layers.

*For the remainder of this pipeline, we will utilize the robust pre-trained `outdoor` weights and rely on dense match filtering (Confidence Thresholding) to isolate anomalies (deforestation).*

In [1]:
import torch
import torch.nn.functional as F

print("❌ EXPERIMENT 1: The Hard Argmax Problem\n" + "-"*40)

heatmap_hard = torch.tensor([[0.1, 0.2, 0.8, 0.3, 0.1]], requires_grad=True)

target_idx = torch.tensor([3.0]) 

try:
    pred_idx_hard = torch.argmax(heatmap_hard, dim=1).float()
    pred_idx_hard.requires_grad_(True) # Forced, otherwise PyTorch immediately stops
    
    loss_hard = F.mse_loss(pred_idx_hard, target_idx)
    loss_hard.backward()
    
    print(f"Gradients on heatmap: {heatmap_hard.grad}")
except Exception as e:
    print(f"PyTorch Error: {e}")
    print("Conclusion: Gradients CANNOT flow backwards through a discrete argmax operation.\n")


print("\n✅ EXPERIMENT 2: The Soft-Argmax Solution\n" + "-"*40)

heatmap_soft = torch.tensor([[0.1, 0.2, 0.8, 0.3, 0.1]], requires_grad=True)
target_idx = torch.tensor([3.0]) 

indices = torch.arange(5).float()

probs = F.softmax(heatmap_soft, dim=1)

pred_idx_soft = torch.sum(probs * indices, dim=1)

print(f"Predicted Continuous Coordinate: {pred_idx_soft.item():.4f}")

loss_soft = F.mse_loss(pred_idx_soft, target_idx)
loss_soft.backward()

print(f"Gradients on heatmap: {heatmap_soft.grad.numpy()}")
print("Conclusion: The computational graph is preserved! Gradients successfully flow back to update the weights.")

❌ EXPERIMENT 1: The Hard Argmax Problem
----------------------------------------
Gradients on heatmap: None

✅ EXPERIMENT 2: The Soft-Argmax Solution
----------------------------------------
Predicted Continuous Coordinate: 2.0183
Gradients on heatmap: [[ 0.6249936   0.3484992   0.01143148 -0.37128413 -0.6136402 ]]
Conclusion: The computational graph is preserved! Gradients successfully flow back to update the weights.
